In [ ]:
from option_analyzer import *
from indicators import compute_emas
self = OptionAnalyzer('quotes', 'chain')

### Run this once every day to load close prices in the past 60 days

In [ ]:
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').tail(60)
df_close.columns.name = 'symbol'
today = pd.Timestamp.now().normalize()
print('df_close data age:', today - df_close.index[-1])

### Either run these two cells

### Or run this cell to read from data directory

In [ ]:
os.system('sync > /dev/null 2>&1')
option_type = 'put'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(['%40s' % _ for _ in map(os.path.basename, latest_option_files)]))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))
dfp = pd.concat([pd.read_csv(_f) for _f in latest_option_files])
df_price = df_close.T.join(dfp.loc[:, ['symbol', 'lastPrice']].drop_duplicates().rename(columns={'lastPrice': today}).set_index('symbol'), how='right').T
df_ema = compute_emas(df_price, [21, 50])

### Put options with no earning date on or before expiration date
- Sell puts to maximize hdteProfit.
- hdte_resid should be < 0.5, maybe even 0.4.
- It's okay to have high spread because half of the spreads have been deducted from hdte profit.

In [ ]:
hdte_resid_ub = 0.4
spread_ub = 25
moneyness_ub = 0.999
premium_lb = 1
delta_lb = -0.2
_filter = (dfp.moneyness <= moneyness_ub) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub)
_filter = _filter & (dfp.mid >= premium_lb) & (dfp.Delta >= delta_lb)
#_filter = _filter & (~dfp.symbol.str.contains('AMD|HOOD|AVGO'))
_filter = _filter & (dfp.E.isna() |(dfp.E > dfp.dte)) # Note: Fidelity's earning report dates are not reliable
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
#_filter = _filter & (dfp.hdteProfit >= 20) & (dfp.strike <= 150)
#_dfp = dfp[_filter].sort_values(by='dth')
print('Options after the filters:', _dfp.shape[0], 'out of', dfp.shape[0])
px.scatter(_dfp.head(200), x='Delta', y='hdteProfit', color='symbol', height=550).show()
df_last_ema = df_price.tail(1).T.join(df_ema.tail(1).T.unstack(level=1).droplevel(level=0, axis=1))
df_last_ema = df_last_ema.loc[list(_dfp.loc[:, 'symbol'].drop_duplicates())]
px.bar(df_last_ema.loc[:, reversed(df_last_ema.columns)], barmode='group').show()
_dfp.head(40)

In [ ]:
_symbol = 'QQQ'
px.line(pd.concat([df_ema[_symbol], df_price[_symbol]], axis=1))

In [ ]:
1165687/589, 350000/585

### Put options for specific symbols

In [ ]:
_filter = dfp.symbol.str.contains('QQQ')#QQQ|SPY|GLD|IBIT|DIA')
_filter = _filter & (dfp.moneyness >= 0.9) & (dfp.Delta >= -0.25) & (dfp.dte <= 60) & (dfp.mid >= 1.3)
#_filter = _filter & (dfp.expDt == '2026-03-31')
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False).head(500)
px.scatter(_dfp, x='Delta', y='hdteProfit', color='expDt', height=500).show()
px.scatter(_dfp[(_dfp.expDt=='2026-03-09')], x='Delta', y='hdteProfit', color='strike', height=500).show()
print(_dfp.shape)
_dfp.head(25)

### Put options: top 500 in-the-money

In [ ]:
_df = dfp[(dfp.moneyness <= 1)].sort_values(by='hdteProfit', ascending=False).head(100)
px.scatter(_df, x='hdte_resid', y='hdteProfit', color='symbol', height=600).show()
_df.head(60)